02 — Dados ADHD-200
Este notebook prepara os dados do ADHD-200 utilizados no experimento de TDAH.
Os dados utilizados correspondem aos arquivos de rs-fMRI pré-processados pelo pipeline Athena e representados utilizando o atlas CC200.
Objetivos
1. Verificar a presença dos arquivos de entrada.
2. Criar a estrutura de diretórios necessária.
3. Extrair o arquivo compactado do ADHD-200.
4. Verificar os dados extraídos.

In [3]:
from pathlib import Path
import sys
import shutil

# ---------------------------------------------------------
# 1. Localizar a raiz do projeto
# ---------------------------------------------------------

def encontrar_raiz_projeto():
    """Procura a pasta que contém o config.py."""
    candidatos = [Path.cwd(), *Path.cwd().parents]

    for caminho in candidatos:
        if (caminho / "config.py").exists():
            return caminho.resolve()

    raise FileNotFoundError(
        "Não foi possível localizar a raiz do projeto. "
        "Verifique se o notebook está dentro do projeto TCC_TDAH."
    )


RAIZ = encontrar_raiz_projeto()

if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import config

config.criar_pastas()

print("=" * 60)
print("CONFIGURAÇÃO DO PROJETO")
print("=" * 60)
print(f"Raiz do projeto: {RAIZ}")
print(f"Dados brutos:    {config.RAW}")
print(f"Dados extraídos: {config.EXTRACTED}")


CONFIGURAÇÃO DO PROJETO
Raiz do projeto: /home/maria/TCC_TDAH
Dados brutos:    /home/maria/TCC_TDAH/data/raw
Dados extraídos: /home/maria/TCC_TDAH/data/extracted


In [4]:
# ---------------------------------------------------------
# 2. Localizar os dados ADHD-200 já extraídos
# ---------------------------------------------------------

NOME_PASTA_DADOS = "ADHD200_CC200_TCs_filtfix"

pasta_raw = config.RAW / NOME_PASTA_DADOS
pasta_extracted = config.EXTRACTED / NOME_PASTA_DADOS

print("=" * 60)
print("LOCALIZAÇÃO DOS DADOS ADHD-200")
print("=" * 60)

if pasta_extracted.exists() and pasta_extracted.is_dir():
    print("✓ Dados já estão em data/extracted:")
    print(f"  {pasta_extracted}")

elif pasta_raw.exists() and pasta_raw.is_dir():
    print("✓ Dados encontrados em data/raw:")
    print(f"  {pasta_raw}")
        
    # Move apenas se ainda não existir em extracted.
    # Não baixa, descompacta ou altera os arquivos internos.
    shutil.move(str(pasta_raw), str(pasta_extracted))

    print("✓ Pasta organizada em data/extracted:")
    print(f"  {pasta_extracted}")

else:
    raise FileNotFoundError(
        "\nPasta dos dados ADHD-200 não encontrada.\n\n"
        f"Procurado em:\n"
        f"  {pasta_raw}\n"
        f"  {pasta_extracted}\n\n"
        "Verifique se a pasta ADHD200_CC200_TCs_filtfix está em "
        "data/raw/."
    )


LOCALIZAÇÃO DOS DADOS ADHD-200
✓ Dados encontrados em data/raw:
  /home/maria/TCC_TDAH/data/raw/ADHD200_CC200_TCs_filtfix
✓ Pasta organizada em data/extracted:
  /home/maria/TCC_TDAH/data/extracted/ADHD200_CC200_TCs_filtfix


In [5]:
# ---------------------------------------------------------
# 3. Localizar e padronizar o arquivo de fenótipos
# ---------------------------------------------------------

NOME_PHENO = "adhd200_preprocessed_phenotypics.tsv"
pheno_padrao = config.RAW / NOME_PHENO

print("=" * 60)
print("ARQUIVO DE FENÓTIPOS")
print("=" * 60)

if pheno_padrao.exists():
    print("✓ Arquivo de fenótipos encontrado:")
    print(f"  {pheno_padrao}")

else:
    candidatos = [
        p for p in config.RAW.glob("*.tsv")
        if p.name != NOME_PHENO
    ]

    if not candidatos:
        raise FileNotFoundError(
            "\nNenhum arquivo .tsv foi encontrado em data/raw/.\n"
            "O arquivo esperado é:\n"
            f"  {pheno_padrao}"
        )

    if len(candidatos) > 1:
        print("Arquivos TSV encontrados:")
        for p in candidatos:
            print(f"  - {p.name}")
        print("\nSerá utilizado o primeiro candidato encontrado.")

    origem = candidatos[0]
        
    shutil.copy2(origem, pheno_padrao)
        
    print("✓ Arquivo de fenótipos localizado:")
    print(f"  {origem}")
    print("\n✓ Cópia padronizada criada:")
    print(f"  {pheno_padrao}")


ARQUIVO DE FENÓTIPOS
✓ Arquivo de fenótipos encontrado:
  /home/maria/TCC_TDAH/data/raw/adhd200_preprocessed_phenotypics.tsv


In [6]:
# ---------------------------------------------------------
# 4. Verificar a estrutura dos dados
# ---------------------------------------------------------

arquivos = [p for p in pasta_extracted.rglob("*") if p.is_file()]

print("=" * 60)
print("VERIFICAÇÃO DOS DADOS")
print("=" * 60)

print(f"\nPasta dos dados:")
print(f"  {pasta_extracted}")

print(f"\nQuantidade de arquivos encontrados: {len(arquivos):,}")

if len(arquivos) == 0:
    raise RuntimeError(
        "A pasta ADHD200_CC200_TCs_filtfix foi encontrada, "
        "mas não contém arquivos."
    )

print("\nPrimeiros arquivos encontrados:")

for arquivo in arquivos[:10]:
    print(f"  - {arquivo.relative_to(pasta_extracted)}")

if len(arquivos) > 10:
    print(f"  ... e mais {len(arquivos) - 10:,} arquivos.")

print("\n✓ Estrutura dos dados encontrada.")


VERIFICAÇÃO DOS DADOS

Pasta dos dados:
  /home/maria/TCC_TDAH/data/extracted/ADHD200_CC200_TCs_filtfix

Quantidade de arquivos encontrados: 1,289

Primeiros arquivos encontrados:
  - extract_tcs.sh
  - NeuroIMAGE/NeuroIMAGE_phenotypic.csv
  - NYU/NYU_phenotypic.csv
  - templates/ADHD200_parcellate_200.nii
  - templates/ADHD200_parcellate_200.nii.gz
  - KKI/KKI_phenotypic.csv
  - KKI/KKI_motion.csv
  - OHSU/OHSU_phenotypic.csv
  - NeuroIMAGE/1208586/sfnwmrda1208586_session_1_rest_1_cc200_TCs.1D
  - NeuroIMAGE/1208586/snwmrda1208586_session_1_rest_1_cc200_TCs.1D
  ... e mais 1,279 arquivos.

✓ Estrutura dos dados encontrada.


In [7]:
# ---------------------------------------------------------
# 5. Verificar o arquivo de fenótipos
# ---------------------------------------------------------

import pandas as pd

print("=" * 60)
print("VERIFICAÇÃO DO ARQUIVO DE FENÓTIPOS")
print("=" * 60)

print(f"Arquivo: {pheno_padrao}")

try:
    pheno = pd.read_csv(pheno_padrao, sep="\t")
except Exception as erro:
    raise RuntimeError(
        "Não foi possível ler o arquivo de fenótipos como TSV.\n"
        f"Erro original: {erro}"
    ) from erro

print(f"\nDimensões: {pheno.shape[0]:,} linhas × {pheno.shape[1]:,} colunas")

print("\nColunas disponíveis:")
for coluna in pheno.columns:
    print(f"  - {coluna}")

print("\nPrimeiras 5 linhas:")
display(pheno.head())


VERIFICAÇÃO DO ARQUIVO DE FENÓTIPOS
Arquivo: /home/maria/TCC_TDAH/data/raw/adhd200_preprocessed_phenotypics.tsv

Dimensões: 973 linhas × 19 colunas

Colunas disponíveis:
  - ScanDir ID
  - Site
  - Gender
  - Age
  - Handedness
  - DX
  - Secondary Dx
  - ADHD Measure
  - ADHD Index
  - Inattentive
  - Hyper/Impulsive
  - IQ Measure
  - Verbal IQ
  - Performance IQ
  - Full2 IQ
  - Full4 IQ
  - Med Status
  - QC_Athena
  - QC_NIAK

Primeiras 5 linhas:


,ScanDir ID,Site,Gender,Age,Handedness,DX,Secondary Dx,ADHD Measure,ADHD Index,Inattentive,Hyper/Impulsive,IQ Measure,Verbal IQ,Performance IQ,Full2 IQ,Full4 IQ,Med Status,QC_Athena,QC_NIAK
0,2371032,3,0.0,10.73,1,0,NaN,2,47,55,43,1.0,121.0,119.0,NaN,122.0,1,1.0,1.0
1,2026113,3,0.0,12.99,1,1,NaN,2,90,89,78,1.0,122.0,108.0,NaN,106.0,1,1.0,1.0
2,3434578,3,0.0,8.12,1,0,NaN,2,42,42,43,1.0,85.0,98.0,NaN,89.0,1,1.0,1.0
3,8628223,3,0.0,10.81,1,0,Simple phobia,2,42,49,49,1.0,85.0,86.0,NaN,97.0,1,1.0,1.0
4,1623716,3,0.0,12.65,1,1,NaN,2,87,90,90,1.0,89.0,88.0,NaN,89.0,1,1.0,1.0


In [8]:
# ---------------------------------------------------------
# 6. Resumo final
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("✓ NOTEBOOK 02 CONCLUÍDO")
print("=" * 60)

print("\nDados ADHD-200:")
print(f"  {pasta_extracted}")

print("\nArquivo de fenótipos:")
print(f"  {pheno_padrao}")

print(f"\nArquivos encontrados nos dados: {len(arquivos):,}")
print(f"Registros no arquivo de fenótipos: {len(pheno):,}")

print("\nPróximo notebook:")
print("  03_Exploracao_ADHD200.ipynb")

print("\n✓ Os dados estão prontos para a etapa de exploração.")



✓ NOTEBOOK 02 CONCLUÍDO

Dados ADHD-200:
  /home/maria/TCC_TDAH/data/extracted/ADHD200_CC200_TCs_filtfix

Arquivo de fenótipos:
  /home/maria/TCC_TDAH/data/raw/adhd200_preprocessed_phenotypics.tsv

Arquivos encontrados nos dados: 1,289
Registros no arquivo de fenótipos: 973

Próximo notebook:
  03_Exploracao_ADHD200.ipynb

✓ Os dados estão prontos para a etapa de exploração.
